# Session 3 lab (Colab fallback): Python & Production Engineering

**AI Forward Deployed Engineering · IIT Delhi × Varsity · 24 Oct 2026**

Use this notebook if Codespaces is blocked or won't start. It covers **L1 Contract**, **L2 Service** and
**L3 Observability**. Colab can't run Git or Docker, so do **L4** later in a codespace.

**How it works:** each *YOUR CODE* cell writes a file. Edit the TODOs in the cell, run it, then run the
check cell under it. Runtime → Run all once first.

Everything here is fictional (Example Bank, its customers, accounts and teams). An offline stub model
answers by default, so no API key is needed. Colab files disappear when the session ends:
File → Save a copy in Drive.

In [ ]:
# @title Setup: run once (installs packages, creates folders)
%pip install -q fastapi==0.150.0 pydantic==2.14.1 httpx==0.28.1 pytest==9.1.1
!mkdir -p app tests data
%env FORCE_OFFLINE=1

**Support files** (the model client, data and tests). Run these cells; no changes needed.

In [ ]:
%%writefile app/__init__.py

In [ ]:
%%writefile app/llm.py
"""The LLM client. You don't need to change this file.

Two backends behind one async interface:
- StubLLM (default): offline and deterministic. Most complaints get valid JSON. Three complaints
  get the kinds of bad output real models produce, so you can see your contract catch them.
- LiveLLM: any OpenAI-compatible endpoint, used only when LLM_API_KEY or GEMINI_API_KEY is set
  (as a Codespaces or Colab secret, never in code).

Everything here is fictional: Example Bank, its customers, accounts and teams.
"""

from __future__ import annotations

import asyncio
import json
import os
import time

PROMPT_VERSION = "triage-v1"

# Complaints that get bad model output from the stub, and what goes wrong.
BAD_OUTPUT = {
    "C-013": "wrong_types",   # priority "urgent" instead of 1-4, team "fraud ops" instead of fraud_ops
    "C-014": "missing_field",  # no "team"
    "C-015": "prose_wrapper",  # "Sure! Here is the JSON: {...}"
}

_RULES = [  # (keywords, category, team, priority); first match wins
    (("did not make", "nahi kiya", "unauthorised", "unauthorized", "otp", "scam", "fraud", "never raised", "kyc"),
     "fraud", "fraud_ops", 1),
    (("charged twice", "swiped", "dispute", "refund", "kat gaye"), "card_dispute", "disputes", 2),
    (("locked", "log in", "login", "password", "app crash", "net banking"), "account_access", "digital_support", 3),
    (("fee", "charges", "penalty"), "charges", "branch_services", 3),
]


def _classify(text: str) -> dict:
    t = text.lower()
    for words, category, team, priority in _RULES:
        if any(w in t for w in words):
            break
    else:
        category, team, priority = "other", "branch_services", 4
    summary = text if len(text) <= 120 else text[:117] + "..."
    return {"category": category, "priority": priority, "team": team, "summary": summary}


def _stub_output(complaint_id: str, text: str) -> str:
    out = _classify(text)
    kind = BAD_OUTPUT.get(complaint_id)
    if kind == "wrong_types":
        out.update(priority="urgent", team="fraud ops")
    elif kind == "missing_field":
        out.pop("team")
    if kind == "prose_wrapper":
        return "Sure! Here is the JSON you asked for:\n" + json.dumps(out)
    return json.dumps(out)


class StubLLM:
    """Offline model. `delay` simulates a slow network call (seconds)."""

    name = "stub"
    model = "stub-triage-1"

    def __init__(self, delay: float = 0.0):
        self.delay = delay

    async def complete(self, complaint_id: str, text: str) -> str:
        """Non-blocking: other requests keep running while this one waits."""
        if self.delay:
            await asyncio.sleep(self.delay)
        return _stub_output(complaint_id, text)

    def complete_blocking(self, complaint_id: str, text: str) -> str:
        """Blocking version, like a sync SDK call. Used to show what NOT to do inside async code."""
        if self.delay:
            time.sleep(self.delay)
        return _stub_output(complaint_id, text)


SYSTEM_PROMPT = (
    "You triage bank complaints. Reply with JSON only, with keys: "
    'category (one of "card_dispute", "fraud", "account_access", "charges", "other"), '
    "priority (integer 1-4, 1 = most urgent), "
    'team (one of "disputes", "fraud_ops", "digital_support", "branch_services"), '
    "summary (at most 200 characters, no account numbers)."
)


class LiveLLM:
    """A real model through an OpenAI-compatible endpoint (Gemini by default)."""

    name = "live"

    def __init__(self):
        from openai import AsyncOpenAI  # imported here so the stub works without network

        self.model = os.environ.get("LLM_MODEL", "gemini-3.8-flash")
        self._client = AsyncOpenAI(
            api_key=os.environ.get("LLM_API_KEY") or os.environ["GEMINI_API_KEY"],
            base_url=os.environ.get("LLM_BASE_URL", "https://generativelanguage.googleapis.com/v1beta/openai/"),
            timeout=4,  # below the service's own 5 s budget (TRIAGE_TIMEOUT_S), so the SDK gives up first
            max_retries=1,
        )

    async def complete(self, complaint_id: str, text: str) -> str:
        resp = await self._client.chat.completions.create(
            model=self.model,
            messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}],
            temperature=0,
        )
        return resp.choices[0].message.content or ""


def get_llm():
    """LiveLLM if a key is set (and FORCE_OFFLINE isn't), otherwise StubLLM."""
    has_key = os.environ.get("LLM_API_KEY") or os.environ.get("GEMINI_API_KEY")
    if has_key and os.environ.get("FORCE_OFFLINE") != "1":
        return LiveLLM()
    return StubLLM()

In [ ]:
%%writefile data/complaints.jsonl
{"id": "C-001", "channel": "email", "text": "I was charged twice for the same grocery bill on my debit card. Please refund the extra amount. Account EB-4418203391."}
{"id": "C-002", "channel": "chat", "text": "There are two transactions on my credit card that I did not make, both at 2 am. Please block the card. Account EB-7720019384."}
{"id": "C-003", "channel": "chat", "text": "My mobile banking app is locked after three wrong password attempts. I cannot log in."}
{"id": "C-004", "channel": "email", "text": "Why was a minimum balance penalty fee deducted this month? I kept the balance above the limit. Account EB-1029384756."}
{"id": "C-005", "channel": "phone", "text": "Someone called pretending to be from the bank and asked for my OTP. I think it is a scam and money has gone. Account EB-5566778899."}
{"id": "C-006", "channel": "chat", "text": "Maine yeh transaction nahi kiya, please check. 4999 rupees ka debit hua hai. Account EB-3141592653."}
{"id": "C-007", "channel": "email", "text": "The merchant cancelled my order but the refund never came back to my card. It has been 15 days."}
{"id": "C-008", "channel": "branch", "text": "I want to update the address on my account. The branch asked me to raise a request online."}
{"id": "C-009", "channel": "chat", "text": "ATM se paise nahi nikle par account se paise kat gaye. Please refund. Account EB-2718281828."}
{"id": "C-010", "channel": "email", "text": "Annual card fee charged even though I was told the card is lifetime free."}
{"id": "C-011", "channel": "chat", "text": "Login OTP is not arriving on my phone, so I cannot access net banking."}
{"id": "C-012", "channel": "phone", "text": "I see a dispute raised on my card that I never raised. Who raised it?"}
{"id": "C-013", "channel": "chat", "text": "Unauthorised UPI debit of 25000 rupees from my account last night. Account EB-9081726354."}
{"id": "C-014", "channel": "email", "text": "Late payment charges were added although I paid before the due date."}
{"id": "C-015", "channel": "chat", "text": "My card was swiped at a restaurant twice and I want to dispute the second charge."}
{"id": "C-016", "channel": "email", "text": "Thank you for the quick help at the branch last week."}
{"id": "C-017", "channel": "chat", "text": "App crash ho raha hai login ke baad, kuch bhi open nahi hota."}
{"id": "C-018", "channel": "phone", "text": "A fraud alert SMS came for a card transaction I did make. Please unblock the card."}
{"id": "C-019", "channel": "email", "text": "Cheque book request is pending for two weeks."}
{"id": "C-020", "channel": "chat", "text": "Received a link saying my account will be blocked unless I update KYC. Is this from the bank?"}

In [ ]:
%%writefile tests/test_l1.py
"""L1 Contract checks. Run: bash grade.sh l1"""

import json

import pytest
from pydantic import ValidationError

from app.llm import StubLLM
from app.models import Complaint, Team, TriageResult, parse_triage

GOOD = {"category": "fraud", "priority": 1, "team": "fraud_ops", "summary": "Unauthorised UPI debit"}


def test_complaint_accepts_valid_input():
    c = Complaint(id="C-001", text="Charged twice", channel="chat")
    assert c.channel == "chat"
    assert Complaint(id="C-002", text="x").channel == "email"  # default


@pytest.mark.parametrize("bad", [{"id": "", "text": "x"}, {"id": "C-1", "text": ""}, {"id": "C-1", "text": "x", "channel": "fax"}])
def test_complaint_rejects_bad_input(bad):
    with pytest.raises(ValidationError):
        Complaint(**bad)


def test_valid_model_output_parses():
    r = parse_triage(json.dumps(GOOD))
    assert isinstance(r, TriageResult)
    assert r.team == Team.FRAUD_OPS and r.priority == 1 and r.needs_human is False


@pytest.mark.parametrize(
    "raw",
    [
        json.dumps({**GOOD, "priority": "urgent", "team": "fraud ops"}),   # wrong types / values
        json.dumps({k: v for k, v in GOOD.items() if k != "team"}),        # missing field
        "Sure! Here is the JSON you asked for:\n" + json.dumps(GOOD),      # prose around the JSON
        json.dumps({**GOOD, "priority": 7}),                                # out of range
        json.dumps({**GOOD, "summary": "x" * 201}),                         # too long
        json.dumps({**GOOD, "refund_amount": 25000}),                       # unknown field
    ],
    ids=["wrong-types", "missing-field", "prose-wrapper", "priority-out-of-range", "summary-too-long", "unknown-field"],
)
def test_bad_model_output_goes_to_human_review(raw):
    r = parse_triage(raw)  # must not raise
    assert r.needs_human is True
    assert r.team == Team.HUMAN_REVIEW


def test_stub_bad_outputs_are_caught():
    stub = StubLLM()
    for cid in ("C-013", "C-014", "C-015"):
        raw = stub.complete_blocking(cid, "Unauthorised UPI debit of 25000 rupees")
        assert parse_triage(raw).needs_human, cid

In [ ]:
%%writefile tests/test_l2.py
"""L2 Service checks. Run: bash grade.sh l2"""

import pytest
from fastapi.testclient import TestClient

import app.main as main
from app.llm import StubLLM


@pytest.fixture
def client(monkeypatch):
    monkeypatch.setattr(main, "llm", StubLLM())
    return TestClient(main.app)


def test_healthz(client):
    r = client.get("/healthz")
    assert r.status_code == 200 and r.json() == {"status": "ok"}


def test_triage_routes_a_dispute(client):
    r = client.post("/triage", json={"id": "C-001", "text": "I was charged twice for the same bill", "channel": "email"})
    assert r.status_code == 200
    body = r.json()
    assert body["category"] == "card_dispute" and body["team"] == "disputes" and body["needs_human"] is False


def test_bad_model_output_returns_human_review(client):
    r = client.post("/triage", json={"id": "C-013", "text": "Unauthorised UPI debit of 25000 rupees"})
    assert r.status_code == 200 and r.json()["team"] == "human_review"


def test_invalid_request_is_rejected_by_the_contract(client):
    r = client.post("/triage", json={"id": "C-1", "text": "", "channel": "fax"})
    assert r.status_code == 422  # FastAPI + Pydantic reject it before our code runs


def test_slow_model_times_out_to_human_review(monkeypatch):
    monkeypatch.setattr(main, "llm", StubLLM(delay=1.0))
    monkeypatch.setattr(main, "TIMEOUT_S", 0.1)
    r = TestClient(main.app).post("/triage", json={"id": "C-001", "text": "charged twice"})
    assert r.status_code == 200
    assert r.json()["needs_human"] is True and "timed out" in r.json()["summary"]


def test_openapi_contract_is_published(client):
    spec = client.get("/openapi.json").json()
    assert "/triage" in spec["paths"] and "/healthz" in spec["paths"]
    fields = spec["components"]["schemas"]["TriageResult"].get("properties", {})
    assert {"category", "priority", "team", "summary"} <= fields.keys(), "the contract must publish TriageResult's fields"

In [ ]:
%%writefile tests/test_l3.py
"""L3 Observability checks. Run: bash grade.sh l3"""

import io
import json
import re

import pytest
from fastapi.testclient import TestClient

import app.main as main
from app.llm import StubLLM
from app.logging_setup import mask, setup_logging

REQUIRED = {"ts", "level", "msg", "request_id", "route", "status", "latency_ms", "model", "prompt_version", "outcome"}
FULL_ACCOUNT = re.compile(r"EB-\d{10}")


def test_mask():
    assert mask("acct EB-4418203391 and EB-7720019384") == "acct EB-******3391 and EB-******9384"
    assert mask("no account here") == "no account here"


@pytest.fixture
def logs(monkeypatch):
    monkeypatch.setattr(main, "llm", StubLLM())
    buf = io.StringIO()
    setup_logging(stream=buf)
    client = TestClient(main.app)
    return client, buf


def _lines(buf):
    return [json.loads(line) for line in buf.getvalue().splitlines() if line.strip()]


def test_one_json_line_per_request_with_the_audit_fields(logs):
    client, buf = logs
    client.post("/triage", json={"id": "C-001", "text": "Charged twice. Account EB-4418203391."})
    lines = [x for x in _lines(buf) if x.get("route") == "/triage"]
    assert len(lines) == 1, "expected exactly one log line for the /triage request"
    missing = REQUIRED - lines[0].keys()
    assert not missing, f"log line is missing {sorted(missing)}"
    assert lines[0]["prompt_version"] and lines[0]["outcome"] in {"routed", "human_review"}


def test_request_id_is_echoed_and_logged(logs):
    client, buf = logs
    r = client.post("/triage", json={"id": "C-001", "text": "charged twice"}, headers={"X-Request-ID": "req-abc123"})
    assert r.headers.get("X-Request-ID") == "req-abc123"
    assert any(x.get("request_id") == "req-abc123" for x in _lines(buf))
    r2 = client.get("/healthz")
    assert r2.headers.get("X-Request-ID"), "a request ID must be generated when the caller sends none"


def test_no_full_account_number_in_logs(logs):
    client, buf = logs
    client.post("/triage", json={"id": "C-013", "text": "Unauthorised UPI debit. Account EB-9081726354."})
    assert buf.getvalue(), "nothing was logged"
    assert not FULL_ACCOUNT.search(buf.getvalue()), "an unmasked account number reached the logs"

## L1 Contract (about 12 min)

Complete `Complaint`, `TriageResult` and `parse_triage` (the TODOs). Bad model output must go to a human,
never crash, never be guessed.

In [ ]:
%%writefile app/models.py
# YOUR CODE (L1): edit the TODOs, then run this cell
"""L1 Contract: what goes into the service and what comes out.

Your tasks (about 12 minutes). Run `bash grade.sh l1` to check.
1. Complete `Complaint`: id (1-20 chars), text (1-4000 chars), channel (email/chat/branch/phone, default email).
2. Complete `TriageResult`: category (Category), priority (int 1-4), team (Team),
   summary (at most 200 chars), needs_human (bool, default False). Reject unknown fields.
3. Complete `parse_triage`: validate the model's raw text; if it's invalid, return
   `human_review(...)` instead of crashing.

Hints: `Field(min_length=..., max_length=..., ge=..., le=...)`, `ConfigDict(extra="forbid")`,
`TriageResult.model_validate_json(raw)`, `except ValidationError`.
"""

from __future__ import annotations

from enum import Enum
from typing import Literal  # noqa: F401  (you'll need it for `channel`)

from pydantic import BaseModel, ConfigDict, Field, ValidationError  # noqa: F401


class Category(str, Enum):
    CARD_DISPUTE = "card_dispute"
    FRAUD = "fraud"
    ACCOUNT_ACCESS = "account_access"
    CHARGES = "charges"
    OTHER = "other"


class Team(str, Enum):
    DISPUTES = "disputes"
    FRAUD_OPS = "fraud_ops"
    DIGITAL_SUPPORT = "digital_support"
    BRANCH_SERVICES = "branch_services"
    HUMAN_REVIEW = "human_review"


class Complaint(BaseModel):
    """What the contact-centre system sends us."""

    # TODO (L1.1): id, text, channel


class TriageResult(BaseModel):
    """What we send back. Code downstream routes on this, so it must be exact."""

    # TODO (L1.2): model_config that rejects unknown fields
    # TODO (L1.2): category, priority, team, summary, needs_human


def human_review(reason: str) -> TriageResult:
    """The safe answer when we can't trust the model's output. (Works once TriageResult is done.)"""
    return TriageResult(
        category=Category.OTHER, priority=2, team=Team.HUMAN_REVIEW, summary=reason[:200], needs_human=True
    )


def parse_triage(raw: str) -> TriageResult:
    """Validate the model's raw text. Never crash, never guess: bad output goes to a human."""
    # TODO (L1.3)
    raise NotImplementedError("L1.3: validate `raw` with TriageResult.model_validate_json")

In [ ]:
# Check L1
!python -m pytest -q -p no:cacheprovider tests/test_l1.py 2>&1 | tail -15

## L2 Service (about 12 min)

Complete `/healthz` and `/triage` (async, with a timeout). Colab can't open `/docs` in a browser, so the
tests call the API in-process with FastAPI's `TestClient`, and the *Try it* cell below does the same.

In [ ]:
%%writefile app/main.py
# YOUR CODE (L2, then L3.3-L3.5): edit the TODOs, then run this cell
"""L2 Service (live lab) and L3 request logging (self-paced).

L2 tasks (about 12 minutes). Run `bash grade.sh l2` to check.
1. `GET /healthz` returns {"status": "ok"}.
2. `POST /triage` takes a Complaint, calls the model with a timeout of TIMEOUT_S seconds,
   and returns parse_triage(raw). On timeout, return human_review("model timed out").
Then run it:  uvicorn app.main:app --reload --port 8000   and open the forwarded port's /docs page.

L3 tasks: see SELF-PACED.md.
"""

from __future__ import annotations

import asyncio  # noqa: F401  (you'll need asyncio.timeout)
import os
import time  # noqa: F401  (L3)
import uuid  # noqa: F401  (L3)

from fastapi import FastAPI, Request

from app.llm import PROMPT_VERSION, get_llm  # noqa: F401  (PROMPT_VERSION is for L3)
from app.logging_setup import logger, setup_logging  # noqa: F401  (L3)
from app.models import Complaint, TriageResult, human_review, parse_triage  # noqa: F401

TIMEOUT_S = float(os.environ.get("TRIAGE_TIMEOUT_S", "5"))
llm = get_llm()

app = FastAPI(title="Example Bank complaint triage", version="0.1.0")


# TODO (L3.3): request-logging middleware. See SELF-PACED.md.


@app.get("/healthz")
def healthz() -> dict:
    # TODO (L2.1)
    raise NotImplementedError("L2.1")


@app.post("/triage", response_model=TriageResult)
async def triage(complaint: Complaint, request: Request) -> TriageResult:
    # TODO (L2.2): the pattern (fill it in and remove the raise):
    #   try:
    #       async with asyncio.timeout(TIMEOUT_S):
    #           raw = await llm.complete(complaint.id, complaint.text)
    #       result = parse_triage(raw)
    #   except TimeoutError:
    #       result = human_review("model timed out")
    #   return result
    # TODO (L3.4): set request.state.log = {...} before returning. See SELF-PACED.md.
    raise NotImplementedError("L2.2")

In [ ]:
# Check L2
!python -m pytest -q -p no:cacheprovider tests/test_l2.py 2>&1 | tail -15

In [ ]:
# Try it: call your API in-process
!python -c "from fastapi.testclient import TestClient; import app.main as m; c = TestClient(m.app); print(c.get('/healthz').json()); print(c.post('/triage', json={'id': 'C-013', 'text': 'Unauthorised UPI debit'}).json())"

## L3 Observability (self-paced, about 20 min)

Follow **L3** in `SELF-PACED.md`: complete `mask()` and `JsonFormatter` below, then add the middleware
(L3.3), the `request.state.log` fields (L3.4) and `setup_logging()` to the **L2 cell above**, and re-run it.

In [ ]:
%%writefile app/logging_setup.py
# YOUR CODE (L3.1-L3.2): edit the TODOs, then run this cell
"""L3 Observability: one JSON log line per request, with account numbers masked.

Self-paced tasks: see SELF-PACED.md. Run `bash grade.sh l3` to check.
"""

from __future__ import annotations

import json  # noqa: F401
import logging
import re
import sys
from datetime import datetime, timezone  # noqa: F401

logger = logging.getLogger("triage")

ACCOUNT_RE = re.compile(r"\bEB-\d{10}\b")  # Example Bank account numbers (fictional format)
FIELDS = ("request_id", "route", "status", "latency_ms", "model", "prompt_version", "outcome", "complaint_excerpt")


def mask(text: str) -> str:
    """EB-4418203391 -> EB-******3391"""
    # TODO (L3.1)
    raise NotImplementedError("L3.1")


class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        # TODO (L3.2): build a dict with ts, level, msg, plus any FIELDS present on the record;
        #   return it as one JSON line, passed through mask().
        raise NotImplementedError("L3.2")


def setup_logging(stream=None, level: str = "INFO") -> logging.Logger:
    """Send the 'triage' logger's output to `stream` (default stdout) as JSON lines."""
    handler = logging.StreamHandler(stream or sys.stdout)
    handler.setFormatter(JsonFormatter())
    logger.handlers = [handler]
    logger.setLevel(level)
    logger.propagate = False
    return logger

In [ ]:
# Check L3
!python -m pytest -q -p no:cacheprovider tests/test_l3.py 2>&1 | tail -15

## L4 Ship: do it in a codespace

Git branches, PRs and Docker need a real environment. Open the lab repo in Codespaces and follow **L4** in
`SELF-PACED.md`. Copy your finished `app/` files across (File → Download, or paste them).